In [5]:
from pathlib import Path
import json
import pandas as pd


def find_wandb_dir(start=Path.cwd()):
    for base in [start] + list(start.parents):
        cand = base / "wandb"
        if cand.exists() and cand.is_dir():
            return cand
    return Path("wandb")


def _parse_config_value(text, key):
    lines = text.splitlines()
    for i, line in enumerate(lines):
        if line.strip() != f"{key}:":
            continue
        for follow in lines[i + 1:]:
            if follow.strip().startswith("value:"):
                return follow.split("value:", 1)[1].strip().strip("'\"")
        return None
    return None


def collect_epochs(wandb_dir=None):
    wandb_dir = Path(wandb_dir) if wandb_dir else find_wandb_dir()
    records = []
    for cfg in wandb_dir.rglob("config.yaml"):
        if cfg.parent.name != "files":
            continue
        text = cfg.read_text()
        dataset = _parse_config_value(text, "dataset")
        seed = _parse_config_value(text, "seed")
        n_epochs = _parse_config_value(text, "n_epochs")
        if n_epochs is None:
            n_epochs = _parse_config_value(text, "epochs")
        if dataset is None or seed is None:
            continue

        summary_path = cfg.parent / "wandb-summary.json"
        epoch_used = None
        if summary_path.exists():
            epoch_used = json.loads(summary_path.read_text()).get("epoch")

        records.append({
            "dataset": dataset,
            "seed": int(seed),
            "max_epochs": int(n_epochs) if n_epochs is not None else None,
            "epochs_used": epoch_used,
        })

    return records


records = collect_epochs()
# Example: check epoch used for dataset/seed
pd.DataFrame([r for r in records if r["dataset"] == "full_merged" and r["seed"] == 40])

# All seeds for a dataset
pd.DataFrame([r for r in records if r["dataset"] == "full_merged"]).sort_values(["seed"])


,dataset,seed,max_epochs,epochs_used
3,full_merged,40,50,43
1,full_merged,94,50,26
0,full_merged,673,50,30
2,full_merged,1899,50,49
7,full_merged,2100,50,42
6,full_merged,2149,50,23
8,full_merged,2230,50,36
5,full_merged,6013,50,46
9,full_merged,9595,50,36
4,full_merged,9898,50,43
